# SmolLM2-1.7B text-to-SQL benchmark (BIRD mini-dev + Spider)

Runs the team harness (`bench/`, branch `kamalesh/smollm2-benchmark`) on a Colab GPU with the same
Ollama model as a local run (`smollm2:1.7b`, Q8_0, 8k context).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

Everything (predictions, LLM cache, data) is kept in Google Drive under `MyDrive/smollm2_bench`,
so if Colab disconnects, just run all again: finished questions are skipped.

Protocol (same as the team): every setup is chosen on the **validation** sets (`bird_val` 98,
`spider_val` 100). The test sets (`bird_test` 155, `spider_test` 200) are run only for the baseline
and the setups picked on validation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/smollm2_bench'
os.makedirs(DRIVE, exist_ok=True)

## 1. Code, Python packages, Ollama

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b kamalesh/smollm2-benchmark https://github.com/sumonesphantom/CSE-572_Project_Skill_Evo.git repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests
OLLAMA_ENV = {**os.environ, 'OLLAMA_NUM_PARALLEL': '2', 'OLLAMA_CONTEXT_LENGTH': '8192', 'OLLAMA_KEEP_ALIVE': '-1'}

def ensure_ollama():
    # start `ollama serve` if it is not answering (called before every run cell)
    try:
        requests.get('http://localhost:11434/api/version', timeout=5); return
    except Exception:
        pass
    subprocess.Popen(['ollama', 'serve'], env=OLLAMA_ENV, stdout=open('/content/ollama.log', 'a'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            print('ollama', requests.get('http://localhost:11434/api/version', timeout=5).json()); return
        except Exception:
            time.sleep(1)
    raise RuntimeError('ollama did not start; see /content/ollama.log')

ensure_ollama()
!ollama pull smollm2:1.7b 2>&1 | tail -1
!ollama show smollm2:1.7b | head -8

## 2. Data (cached in Drive after the first run)

In [ ]:
%cd /content/repo
import pathlib
RAW = pathlib.Path(DRIVE) / 'raw'
if not (RAW / 'bird/minidev/MINIDEV/mini_dev_sqlite.json').exists():
    !mkdir -p {RAW}/bird {RAW}/spider
    !curl -sSL -o /content/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip && unzip -q -o /content/minidev.zip -d {RAW}/bird && rm /content/minidev.zip
    !gdown -q 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O /content/spider.zip && unzip -q -o /content/spider.zip -d {RAW}/spider && rm /content/spider.zip
# copy to local disk: SQLite on Google Drive is slow
!mkdir -p data/raw && rsync -a {RAW}/ data/raw/
# card_games ships in WAL mode, which cannot be opened read-only (52 questions silently dropped otherwise)
import sqlite3, glob
for p in glob.glob('data/raw/bird/minidev/MINIDEV/dev_databases/*/*.sqlite'):
    c = sqlite3.connect(p)
    if c.execute('PRAGMA journal_mode').fetchone()[0].lower() == 'wal':
        print('fixed WAL:', p, c.execute('PRAGMA journal_mode=delete').fetchone())
    c.close()
!python scripts/prepare_data.py | grep -E "tasks|dropped"
!git status --short data/   # must print nothing: the committed splits are unchanged

In [ ]:
# keep predictions and the LLM cache in Drive so a disconnect loses nothing
%cd /content/repo
for local, remote in (('runs/bench', f'{DRIVE}/runs_bench'), ('data/cache', f'{DRIVE}/cache')):
    os.makedirs(remote, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    !rm -rf {local} && ln -s {remote} {local}
!python -m pytest -q tests 2>&1 | tail -1

## 3. Smoke test (3 questions per set)

In [ ]:
ensure_ollama()
!python -m bench.run --model smollm2:1.7b --strategy sm_pruned --sets bird_val,spider_val --num-ctx 8192 --limit 3 --no-ves --workers 2

## 4. Selection runs on the validation sets

Rungs (each changes one thing; see `bench/strategies.py`):

| Rung | Strategy | Adds |
|---|---|---|
| P1 | zs_repo | repo prompt (team baseline) |
| S1 | sm_extract | + robust SQL extraction (same prompt, reuses P1's answers) |
| S2 | sm_compact | + compact full schema, no 6,000-character cut |
| S3 | sm_pruned | + schema linking: top-4 tables + BIRD column descriptions |
| S4 | sm_pruned_rules | + output rules |
| S5 / S6 | sm_pruned_fs1 / fs3 | + 1 / 3 retrieved solved examples |
| S7 | sm_pruned_sc5 | + best-of-5 vote on execution results |
| S8 / S9 | sm_pruned_fs1_sc5 / fs3_sc5 | examples + vote |
| P2 / P5m | zs_mschema / ms_fs3 | the Qwen branch's M-Schema rungs, for comparison |

In [ ]:
VAL = 'bird_val,spider_val'
RUNGS = ['zs_repo', 'sm_extract', 'sm_compact', 'sm_pruned', 'sm_pruned_rules', 'sm_pruned_fs1', 'sm_pruned_fs3',
         'sm_pruned_sc5', 'sm_pruned_fs1_sc5', 'sm_pruned_fs3_sc5', 'zs_mschema', 'ms_fs3']
# one rung per call, restarting Ollama if it died, so a server crash cannot stop the remaining rungs
for r in RUNGS:
    ensure_ollama()
    !python -m bench.run --model smollm2:1.7b --strategy {r} --sets {VAL} --num-ctx 8192 --workers 2

In [ ]:
# Ollama problems during the runs (empty = none)
!grep -iE "error|panic|out of memory|cuda|fatal" /content/ollama.log | tail -20

In [ ]:
import json, pandas as pd
rows = []
for r in RUNGS:
    mf = f'runs/bench/smollm2-1.7b__{r}/metrics.json'
    if not os.path.exists(mf):
        print('missing (rerun the cell above):', r); continue
    m = json.load(open(mf))['metrics']
    if not all(s in m and m[s] for s in ('bird_val', 'spider_val')):
        print('incomplete (rerun the cell above):', r); continue
    rows.append({'strategy': r, **{f'{s} EX': round(100 * m[s]['ex'], 1) for s in ('bird_val', 'spider_val')},
                 'ctx overflow %': round(100 * max(m[s].get('ctx_overflow_rate', 0) for s in m if m[s]), 1),
                 'LLM errors': sum(m[s].get('errors', {}).get('llm_error', 0) for s in m if m[s]),
                 's/q': round(m['bird_val'].get('avg_llm_seconds') or 0, 1)})
val = pd.DataFrame(rows)
val['mean val EX'] = (val['bird_val EX'] + val['spider_val EX']) / 2
display(val.sort_values('mean val EX', ascending=False))
best_bird = val.loc[val['bird_val EX'].idxmax(), 'strategy']
best_spider = val.loc[val['spider_val EX'].idxmax(), 'strategy']
best_mean = val.loc[val['mean val EX'].idxmax(), 'strategy']
print('picked on validation -> BIRD:', best_bird, '| Spider:', best_spider, '| overall:', best_mean)

## 5. Test sets, run once: baseline + the setups picked on validation

In [ ]:
FINAL = list(dict.fromkeys(['zs_repo', 'sm_pruned', best_bird, best_spider, best_mean]))
print('test runs:', FINAL)
for r in FINAL:
    ensure_ollama()
    !python -m bench.run --model smollm2:1.7b --strategy {r} --sets bird_val,spider_val,bird_test,spider_test --num-ctx 8192 --workers 2

## 6. Report: team CSV, tables, per-question outputs → download

In [ ]:
import json
json.dump({'picked': {'bird': best_bird, 'spider': best_spider, 'overall': best_mean}, 'final_runs': FINAL},
          open('selection.json', 'w'), indent=2)
!python -m bench.smol_report && cp selection.json results/smollm2/
!cat results/smollm2/tables.md
!cd results && zip -qr /content/smollm2_results.zip smollm2 && cp /content/smollm2_results.zip {DRIVE}/
from google.colab import files
files.download('/content/smollm2_results.zip')